In [43]:
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langchain_core.tools import InjectedToolArg
from dotenv import load_dotenv
from typing import Annotated
import requests
load_dotenv()

True

In [44]:
llm=ChatGroq(model="llama-3.3-70b-versatile", temperature=0.9)

In [99]:
@tool 
def currency_conversion(base_currency: str, target_currency: str) -> float:
    """Fetch the currency conversion factor between a given base currency and target currency."""
    API_KEY="b3b894741b269000b4f6b440"
    url=f"https://v6.exchangerate-api.com/v6/{API_KEY}/pair/{base_currency}/{target_currency}"
    response=requests.get(url)
    return response.json()
@tool
def convert_currency(amount:int, conversion_factor: Annotated[float, InjectedToolArg]) -> float:
    """Convert the given amount using the provided conversion factor."""
    return amount * conversion_factor


In [27]:
convert_currency.invoke({"amount": 100, "conversion_factor": 82.5})

8250.0

In [100]:
llm_with_tools=llm.bind_tools([currency_conversion, convert_currency])

In [19]:
result=currency_conversion.invoke({"base_currency": "USD", "target_currency": "INR"})

In [101]:
query=HumanMessage(content="What is the conversion factor between USD and PKR and based on that convert 100 USD to PKR?")
messages=[query]

In [102]:
ai_message=llm_with_tools.invoke(messages)
messages.append(ai_message)


In [77]:
ai_message

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'efm36rq1f', 'function': {'arguments': '{"base_currency":"USD","target_currency":"PKR"}', 'name': 'currency_conversion'}, 'type': 'function'}, {'id': 'b8kk3d92a', 'function': {'arguments': '{"amount":100}', 'name': 'convert_currency'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 317, 'total_tokens': 354, 'completion_time': 0.101248533, 'completion_tokens_details': None, 'prompt_time': 0.05748372, 'prompt_tokens_details': None, 'queue_time': 0.015741853, 'total_time': 0.158732253}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dd291-325c-7151-b291-03dd146bd710-0', tool_calls=[{'name': 'currency_conversion', 'args': {'base_currency': 'USD', 'target_currency': 'PKR'}, 'id': 'efm36rq1f', 'type': 'tool_call'}, {'name': 'co

In [103]:
ai_message.tool_calls


[{'name': 'currency_conversion',
  'args': {'base_currency': 'USD', 'target_currency': 'PKR'},
  'id': 'yraz8sj67',
  'type': 'tool_call'},
 {'name': 'convert_currency',
  'args': {'amount': 100},
  'id': 'sedcn1wnk',
  'type': 'tool_call'}]

In [104]:
import json
for tools in ai_message.tool_calls:
    #get first tool currency_conversion
    if tools["name"] == "currency_conversion":
        tool_message1=currency_conversion.invoke(tools)
        conversation_rate=json.loads(tool_message1.content)["conversion_rate"]
        messages.append(tool_message1)
        print(f"Conversion rate between USD and PKR is {conversation_rate}")
    if tools["name"] == "convert_currency":
        tools["args"]["conversion_factor"]=conversation_rate
        tool_message2=convert_currency.invoke(tools)
        messages.append(tool_message2)
        

Conversion rate between USD and PKR is 278.79


In [105]:
messages

[HumanMessage(content='What is the conversion factor between USD and PKR and based on that convert 100 USD to PKR?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'yraz8sj67', 'function': {'arguments': '{"base_currency":"USD","target_currency":"PKR"}', 'name': 'currency_conversion'}, 'type': 'function'}, {'id': 'sedcn1wnk', 'function': {'arguments': '{"amount":100}', 'name': 'convert_currency'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 317, 'total_tokens': 354, 'completion_time': 0.084796771, 'completion_tokens_details': None, 'prompt_time': 0.080255683, 'prompt_tokens_details': None, 'queue_time': 0.160372864, 'total_time': 0.165052454}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dd2a0-9388-76e3-b05e-19ac0f87ca

In [107]:
llm_with_tools.invoke(messages).content

'The conversion factor between USD and PKR is 278.79. Based on this conversion factor, 100 USD is equivalent to 27879 PKR.'